# CoreForge — Hindi AI v0.3 Lite (reliable T4 fallback)
This is a **new alternative notebook**, not the failing Qwen3-4B notebook. The goal is to finish an actual research checkpoint on free Colab T4, with clear errors and a ZIP output.

**Model:** Qwen3-1.7B (known to have trained successfully for v0.2), LoRA 4-bit. We are not claiming it is more accurate than v0.2. **No automatic deployment.**
This version uses the existing CoreForge data/trainer with stronger diagnostics, a modest resource budget and explicit manual gates.


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
print("GPU:",torch.cuda.get_device_name(0))
print("VRAM (GiB):",round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print("Free disk (GiB):",round(shutil.disk_usage('/content').free/2**30,1))
assert shutil.disk_usage('/content').free>12*2**30,"Not enough free Colab disk"


## 1 — Install libraries and clone current training code
Run this cell first. Installs the same training dependencies used in v0.2. Each command is checked; a failure stops immediately.


In [ ]:
def checked(cmd, title):
    print("\n###",title,flush=True)
    subprocess.run(cmd,check=True)
repo=Path('/content/coreforge-lite')
if not repo.exists():
    checked(['git','clone','--depth','1','https://github.com/ajay333X/my-adatacore-site.git',str(repo)],'Get CoreForge scripts')
else:
    checked(['git','-C',str(repo),'pull','--ff-only'],'Update CoreForge scripts')
os.chdir(repo)
checked([sys.executable,'-m','pip','install','-q','unsloth','datasets>=3,<5','transformers>=4.51,<5','peft>=0.15,<1','accelerate>=1.4,<2'], 'Install packages')
print('Ready:',repo)


## 2 — Prepare licensed Hindi/Hinglish training data
Starts with 1,500 candidates from each previously verified public source (the scale that worked in v0.2). Then filters repetitive content and duplicated prompts. The filter also produces a manual-review CSV. Source sizes can vary.


In [ ]:
RAW=Path('/content/coreforge-lite-hindi-raw')
DATA=Path('/content/coreforge-lite-hindi-filtered')
checked([sys.executable,'workers/coreforge_hindi_data.py','--sources','aya_hindi,hinglish','--limit','1500','--accept-license','--out',str(RAW)],'Download and preprocess Hindi data')
checked([sys.executable,'workers/coreforge_hindi_v03_filter.py','--input',str(RAW),'--output',str(DATA),'--sample','40'],'Filter duplicates and repetition')
import json
manifest=json.loads((DATA/'manifest.json').read_text())
print('Examples:',manifest['splits'])
assert manifest['splits']['train']>100, 'Insufficient training examples'


## 3 — Inspect data (required)
Download the manual sample and read it before proceeding. Check for bad Hindi, irrelevant answers, repeated content, personal information and licensing concerns. A sample does not prove the entire corpus is clean.


In [ ]:
from google.colab import files
files.download(str(DATA/'manual_review.csv'))


In [ ]:
I_REVIEWED_SAMPLE=False # Change to True only after examining manual_review.csv
if not I_REVIEWED_SAMPLE:
    raise RuntimeError('Stop: please review the downloaded data sample before training.')


## 4 — Train Qwen3-1.7B LoRA on T4
This uses a conservative 512-token sequence length and 120 optimizer steps to reduce resource failures. Unlike the prior notebook, a subprocess error is surfaced at the training step, not hidden until ZIP creation. It saves an adapter, not a full base model.


In [ ]:
OUT=Path('/content/coreforge-hindi-v03-lite-checkpoint')
checked([sys.executable,'workers/coreforge_chat_train.py','--data',str(DATA),'--output',str(OUT),
         '--model','Qwen/Qwen3-1.7B','--max-tokens','512','--max-steps','120',
         '--approve-research-run'],'Train v0.3 Lite LoRA')
report_path=OUT/'training_report.json'
adapter_weights=OUT/'adapter'/'adapter_model.safetensors'
if not report_path.is_file() or not adapter_weights.is_file():
    raise RuntimeError('Trainer returned but did not write a complete checkpoint. Review training log above.')
report=json.loads(report_path.read_text())
print('Training finished successfully:',report['base_model'])
print('Validation loss:',report.get('validation_metrics',{}).get('eval_loss'))


## 5 — Download the trained model ZIP
The download cell only runs when actual adapter weights and report exist. Keep the ZIP; Colab sessions are temporary. **Do not deploy this research checkpoint until it passes a response-quality comparison.**


In [ ]:
import zipfile
assert adapter_weights.is_file() and report_path.is_file(), 'No finished checkpoint. Finish Step 4 first.'
archive=Path('/content/Adatacore-Hindi-AI-v0.3-Lite.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in (OUT/'adapter').rglob('*'):
        if file.is_file():z.write(file,arcname=str(file.relative_to(OUT.parent)))
    z.write(report_path,arcname=str(report_path.relative_to(OUT.parent)))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None,'ZIP integrity failed'
print('Verified ZIP:',archive, 'size MB:',round(archive.stat().st_size/1e6,1))
files.download(str(archive))
files.download(str(report_path))


## After download
Upload the ZIP and JSON training report to ChatGPT. The next step is to test Hindi answers against the unmodified **same** Qwen3-1.7B model. Low loss alone is not a measure of correctness. If training fails, share the **first error shown in Step 4**. This fallback does not replace the stronger 4B experiment.
